### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ecommerce_shipping",
    dataset_year="2021",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/prachi13/customer-analytics",
    download_description="""
kaggle datasets download prachi13/customer-analytics -p local-data-warehouse/ecommerce_shipping/ && unzip local-data-warehouse/ecommerce_shipping/customer-analytics.zip -d local-data-warehouse/ecommerce_shipping/ && rm local-data-warehouse/ecommerce_shipping/customer-analytics.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{gopalani2021ecommerce,
  author       = {Prachi Gopalani},
  title        = {E-Commerce Shipping Data},
  year         = {2021},
  howpublished = {\url{https://www.kaggle.com/datasets/prachi13/customer-analytics}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="gopalani2021ecommerce",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We dropped the ID column.
- We renamed the target feature "Reached.on.Time_Y.N" to "ArrivedLate" and mapped binary values to "Yes"/"No".
- Anomaly: the target and task seems somewhat disconnected from the features. Moreover, some source information on the data is missing and there might be some translation issues.
- Anomaly: there might be some data issues related to "Warehouse_block" and the value "F" consisting of two block "E" and "F".
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="ArrivedLate",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="ArrivedLate",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/Train.csv")

target_feature = "ArrivedLate"
df = df.rename(columns={"Reached.on.Time_Y.N": target_feature})
df = df.drop(columns=["ID"])
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"})

cat_features = [
    "Warehouse_block",
    "Mode_of_Shipment",
    "Product_importance",
    "Gender",
    "ArrivedLate",
]

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()